# Marketing Mix Model — Full Pipeline Notebook**Omar Oudrari | March 2026 | Nova DTC Electronics**This notebook runs the complete MMM pipeline end-to-end:1. **Setup** — Import libraries and define parameters2. **Data Generation** — Create 104-week synthetic dataset3. **EDA** — Explore distributions, correlations, and time patterns4. **Transformations** — Apply adstock + Hill saturation pipeline5. **Regression** — OLS model with diagnostics6. **Decomposition** — Revenue attribution waterfall + ROI7. **Optimization** — Budget reallocation with scipy.optimize8. **Results** — Scenario comparison and key findings> **Run top-to-bottom after Kernel → Restart.** Every cell depends on cells above it.

---## 1. Setup — Libraries & Parameters

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats, optimize
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)
np.random.seed(42)

print("All imports successful.")

### Model ParametersValidated against Meta Robyn, Google Meridian, Joseph MPRA 7683, PyMC-Marketing, and Analytic Partners ROI Genome.

In [ ]:
# ── Channel Parameters ──
CHANNEL_PARAMS = {
    'tv':      {'mean': 40000, 'std': 10000, 'decay': 0.75, 'K': 0.50, 'S': 2.0, 'coeff': 0.15},
    'search':  {'mean': 20000, 'std': 5000,  'decay': 0.20, 'K': 0.30, 'S': 1.2, 'coeff': 0.25},
    'social':  {'mean': 15000, 'std': 4000,  'decay': 0.45, 'K': 0.40, 'S': 1.8, 'coeff': 0.18},
    'email':   {'mean': 3000,  'std': 800,   'decay': 0.20, 'K': 0.30, 'S': 1.0, 'coeff': 0.10},
    'display': {'mean': 10000, 'std': 3000,  'decay': 0.30, 'K': 0.35, 'S': 1.5, 'coeff': 0.12},
}

BASE_REVENUE = 200_000
SEASONALITY_AMP = 0.20
NOISE_STD = 0.10
N_WEEKS = 104
CHANNELS = list(CHANNEL_PARAMS.keys())

HOLIDAY_EFFECTS = {'black_friday': 0.75, 'christmas': 0.60, 'memorial_day': 0.20,
                   'july_4th': 0.20, 'labor_day': 0.15, 'easter': 0.10}
HOLIDAY_WEEKS = {'easter': [14,66], 'memorial_day': [21,73], 'july_4th': [26,78],
                 'labor_day': [35,87], 'black_friday': [47,99], 'christmas': [51,103]}
COMP_EFFECT = -0.08
COMP_PROB = 0.15

print("Parameters loaded.")

---## 2. Core Transformation FunctionsThe two building blocks of any MMM: **adstock** (carryover) and **Hill saturation** (diminishing returns).

In [ ]:
def adstock_geometric(spend, decay):
    """Geometric adstock: adstocked[t] = spend[t] + decay * adstocked[t-1]"""
    out = np.zeros(len(spend))
    out[0] = spend[0]
    for t in range(1, len(spend)):
        out[t] = spend[t] + decay * out[t-1]
    return out

def hill_saturation(x, K, S):
    """Hill function: models diminishing returns. Output in [0, 1]."""
    x = np.maximum(x, 1e-10)
    return 1.0 / (1.0 + (K / x) ** S)

def normalize(arr):
    """Scale to [0, 1]."""
    mn, mx = arr.min(), arr.max()
    return np.zeros(len(arr)) if mx == mn else (arr - mn) / (mx - mn)

def apply_pipeline(spend, decay, K, S):
    """Full pipeline: raw → adstock → normalize → Hill saturation."""
    adstocked = adstock_geometric(spend, decay)
    normed = normalize(adstocked)
    saturated = hill_saturation(normed, K, S)
    return adstocked, normed, saturated

print("Transformation functions defined.")

---## 3. Data GenerationCreate a 104-week synthetic dataset with known ground-truth relationships so the model can be validated.

In [ ]:
# Generate weekly spend per channel
spend_data = {}
for ch, p in CHANNEL_PARAMS.items():
    spend_data[ch] = np.maximum(np.random.normal(p['mean'], p['std'], N_WEEKS), 0)

# Transform spend through pipeline
saturated_data = {}
for ch, p in CHANNEL_PARAMS.items():
    _, _, saturated_data[ch] = apply_pipeline(spend_data[ch], p['decay'], p['K'], p['S'])

# Seasonality (peaks Q4, troughs Q2)
weeks = np.arange(N_WEEKS)
seasonality = SEASONALITY_AMP * np.sin(2 * np.pi * (weeks - 37) / 52)

# Holiday lifts
holiday_lift = np.zeros(N_WEEKS)
for holiday, effect in HOLIDAY_EFFECTS.items():
    for w in HOLIDAY_WEEKS[holiday]:
        if w < N_WEEKS:
            holiday_lift[w] = effect
holiday_flag = (holiday_lift > 0).astype(int)

# Competitor promos (~15% of weeks)
competitor_promo = np.random.binomial(1, COMP_PROB, N_WEEKS)

# Calculate revenue = base × controls + Σ(channel contributions) + noise
multiplier = (1 + seasonality) * (1 + holiday_lift) * (1 + competitor_promo * COMP_EFFECT)
channel_contrib = sum(CHANNEL_PARAMS[ch]['coeff'] * saturated_data[ch] * BASE_REVENUE for ch in CHANNELS)
noise = np.random.normal(0, NOISE_STD * BASE_REVENUE, N_WEEKS)
revenue = np.maximum(BASE_REVENUE * multiplier + channel_contrib + noise, 0)

# Assemble DataFrame
df = pd.DataFrame({
    'week': np.arange(1, N_WEEKS+1),
    **{f'{ch}_spend': spend_data[ch].round(2) for ch in CHANNELS},
    'seasonality': seasonality.round(4),
    'holiday_flag': holiday_flag,
    'holiday_lift': holiday_lift.round(2),
    'competitor_promo': competitor_promo,
    'revenue': revenue.round(2),
})

print(f"Dataset shape: {df.shape}")
print(f"Mean weekly revenue: ${df['revenue'].mean():,.0f}")
print(f"Revenue range: ${df['revenue'].min():,.0f} – ${df['revenue'].max():,.0f}")
df.head()

---## 4. Exploratory Data AnalysisCheck distributions, correlations, and time patterns before modeling.

In [ ]:
# Distribution of spend by channel + revenue
spend_cols = [f'{ch}_spend' for ch in CHANNELS]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#555555']

for i, (col, color) in enumerate(zip(spend_cols + ['revenue'], colors)):
    ax = axes[i // 3, i % 3]
    ax.hist(df[col], bins=20, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(df[col].mean(), color='black', linestyle='--', label=f'Mean: ${df[col].mean():,.0f}')
    ax.set_title(col.replace('_spend', '').replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel('Weekly Spend ($)' if 'spend' in col else 'Weekly Revenue ($)')
    ax.legend(fontsize=8)

plt.suptitle('Distribution of Weekly Spend by Channel + Revenue', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/Distribution_Weekly_Spend_Channel+Revenue.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation matrix
corr_cols = spend_cols + ['seasonality', 'holiday_lift', 'competitor_promo', 'revenue']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
labels = [c.replace('_spend','').replace('_',' ').title() for c in corr_cols]
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels)

for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=8)

plt.colorbar(im)
plt.title('Correlation Matrix — Spend, Controls, and Revenue', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/Correlation_Matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key correlations with revenue:")
for c in corr_cols[:-1]:
    print(f"  {c:25s} r = {corr.loc[c, 'revenue']:+.3f}")

**Key EDA takeaways:**- Seasonality (r=0.69) and holiday lift (r=0.66) are the strongest raw correlations with revenue — controlling for them is essential- Channel spend correlations with revenue are weak (r=0.06–0.24) because the raw relationship is masked by confounders- Low multicollinearity between channels (no pair > 0.27) — good for regression

---## 5. Feature TransformationsApply the adstock → normalize → Hill saturation pipeline to each channel.

In [ ]:
# Apply pipeline to each channel
for ch in CHANNELS:
    p = CHANNEL_PARAMS[ch]
    adstocked, normed, saturated = apply_pipeline(df[f'{ch}_spend'].values, p['decay'], p['K'], p['S'])
    df[f'{ch}_adstocked'] = adstocked
    df[f'{ch}_normalized'] = normed
    df[f'{ch}_saturated'] = saturated

print(f"Transformed dataset shape: {df.shape}")
print(f"\nSaturated feature ranges:")
for ch in CHANNELS:
    s = df[f'{ch}_saturated']
    print(f"  {ch:8s}: {s.min():.4f} – {s.max():.4f}")

---## 6. OLS Regression ModelFit the MMM with 5 saturated channel features + 3 control variables.

In [ ]:
FEATURE_COLS = [f'{ch}_saturated' for ch in CHANNELS] + ['seasonality', 'holiday_lift', 'competitor_promo']
CHANNEL_LABELS = {'tv_saturated': 'TV', 'search_saturated': 'Paid Search', 'social_saturated': 'Social Media',
                  'email_saturated': 'Email', 'display_saturated': 'Display', 'seasonality': 'Seasonality',
                  'holiday_lift': 'Holiday Lift', 'competitor_promo': 'Competitor Promo'}

X = sm.add_constant(df[FEATURE_COLS])
y = df['revenue']
model = sm.OLS(y, X).fit()

print(model.summary())

In [ ]:
# Coefficient signs check — all channel coefficients should be positive
print("\nCoefficient signs:")
for col in [f'{ch}_saturated' for ch in CHANNELS]:
    coef = model.params[col]
    sign = "✅ POSITIVE" if coef > 0 else "❌ NEGATIVE"
    print(f"  {CHANNEL_LABELS[col]:15s}  {coef:>12,.2f}  {sign}")

print(f"\nR-squared:          {model.rsquared:.4f}")
print(f"Adjusted R-squared: {model.rsquared_adj:.4f}")

### VIF — Multicollinearity Check

In [ ]:
X_vif = sm.add_constant(df[FEATURE_COLS])
print("VIF scores (< 5 = OK, 5-10 = concerning, > 10 = serious):")
for i, col in enumerate(X_vif.columns):
    if col == 'const': continue
    vif = variance_inflation_factor(X_vif.values, i)
    status = "OK" if vif < 5 else ("CONCERNING" if vif < 10 else "SERIOUS")
    label = CHANNEL_LABELS.get(col, col)
    print(f"  {label:20s}  VIF = {vif:.2f}  {status}")

### Train/Test Validation

In [ ]:
train_df, test_df = df.iloc[:80], df.iloc[80:]
X_train = sm.add_constant(train_df[FEATURE_COLS])
X_test = sm.add_constant(test_df[FEATURE_COLS])
train_model = sm.OLS(train_df['revenue'], X_train).fit()

train_mape = mean_absolute_percentage_error(train_df['revenue'], train_model.predict(X_train)) * 100
test_mape = mean_absolute_percentage_error(test_df['revenue'], train_model.predict(X_test)) * 100

print(f"Training MAPE: {train_mape:.2f}%")
print(f"Test MAPE:     {test_mape:.2f}%")
print(f"\nInterpretation: {'Strong' if test_mape <= 15 else 'Acceptable' if test_mape <= 20 else 'Weak'} MMM fit.")

---## 7. Model DiagnosticsCheck residual behavior: no pattern, approximate normality, no heteroscedasticity.

In [ ]:
fitted = model.fittedvalues
resid = model.resid
std_resid = (resid - resid.mean()) / resid.std()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Actual vs Predicted
axes[0,0].plot(df['week'], y, label='Actual', lw=1.5)
axes[0,0].plot(df['week'], fitted, label='Predicted', lw=1.5)
axes[0,0].set_title('Actual vs Predicted Revenue', fontweight='bold')
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

# Residuals vs Fitted
axes[0,1].scatter(fitted, resid, alpha=0.6, edgecolors='white', lw=0.5)
axes[0,1].axhline(0, color='red', ls='--')
axes[0,1].set_title('Residuals vs Fitted', fontweight='bold')
axes[0,1].grid(alpha=0.3)

# Q-Q Plot
sm.qqplot(resid, line='45', fit=True, ax=axes[1,0])
axes[1,0].set_title('Q-Q Plot of Residuals', fontweight='bold')

# Standardized Residuals Over Time
axes[1,1].plot(df['week'], std_resid, lw=1.2)
axes[1,1].axhline(0, color='black', ls='--')
axes[1,1].axhline(2, color='red', ls='--', alpha=0.5)
axes[1,1].axhline(-2, color='red', ls='--', alpha=0.5)
axes[1,1].set_title('Standardized Residuals Over Time', fontweight='bold')
axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('images/phase4_diagnostics_grid.png', dpi=150, bbox_inches='tight')
plt.show()

# Statistical tests
bp_stat, bp_p, _, _ = het_breuschpagan(resid, X)
shapiro_stat, shapiro_p = stats.shapiro(resid)
dw = sm.stats.stattools.durbin_watson(resid)
print(f"Breusch-Pagan p-value: {bp_p:.4f} ({'No heteroscedasticity' if bp_p > 0.05 else 'Possible heteroscedasticity'})")
print(f"Shapiro-Wilk p-value:  {shapiro_p:.4f} ({'Normal' if shapiro_p > 0.05 else 'Non-normal tails'})")
print(f"Durbin-Watson:         {dw:.4f} (target ~2.0)")

---## 8. Revenue DecompositionAttribute average weekly revenue to each component: base, channels, controls.

In [ ]:
# Build contribution table
mean_revenue = df['revenue'].mean()
contrib_data = []
contrib_data.append({'component': 'Base Revenue', 'weekly': model.params['const']})

for col in FEATURE_COLS:
    label = CHANNEL_LABELS[col]
    contrib_data.append({'component': label, 'weekly': model.params[col] * df[col].mean()})

contrib_df = pd.DataFrame(contrib_data)
contrib_df['pct'] = (contrib_df['weekly'] / mean_revenue * 100).round(1)

print("Revenue Decomposition — Average Weekly")
print("=" * 55)
for _, row in contrib_df.iterrows():
    print(f"  {row['component']:20s}  ${row['weekly']:>10,.0f}  ({row['pct']:>5.1f}%)")
print(f"  {'TOTAL':20s}  ${contrib_df['weekly'].sum():>10,.0f}")
print(f"  {'Actual Mean':20s}  ${mean_revenue:>10,.0f}")

In [ ]:
# ROI calculation
spend_means = {ch: df[f'{ch}_spend'].mean() for ch in CHANNELS}
channel_contribs = {ch: model.params[f'{ch}_saturated'] * df[f'{ch}_saturated'].mean() for ch in CHANNELS}

print("Channel ROI — Average Weekly Revenue per $1 Spent")
print("=" * 50)
roi_data = []
for ch in CHANNELS:
    roi = channel_contribs[ch] / spend_means[ch] if spend_means[ch] > 0 else 0
    roi_data.append({'channel': CHANNEL_LABELS[f'{ch}_saturated'], 'roi': roi,
                     'spend': spend_means[ch], 'contribution': channel_contribs[ch]})
    print(f"  {CHANNEL_LABELS[f'{ch}_saturated']:15s}  ROI = {roi:.2f}x  (${channel_contribs[ch]:,.0f} / ${spend_means[ch]:,.0f})")

roi_df = pd.DataFrame(roi_data).sort_values('roi', ascending=True)

In [ ]:
# ROI Bar Chart
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#10B981' if r >= 1 else '#F59E0B' for r in roi_df['roi']]
bars = ax.barh(roi_df['channel'], roi_df['roi'], color=colors, edgecolor='white', height=0.6)
ax.axvline(1.0, color='gray', ls='--', alpha=0.5, label='Breakeven (1.0x)')
avg_roi = sum(channel_contribs.values()) / sum(spend_means.values())
ax.axvline(avg_roi, color='gray', ls='--', alpha=0.7, label=f'Average ROI = {avg_roi:.2f}x')

for bar, roi in zip(bars, roi_df['roi']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, f'{roi:.2f}x',
            va='center', fontweight='bold', fontsize=11)

ax.set_xlabel('Average ROI (Revenue Contribution / Raw Spend)')
ax.set_title('Channel ROI Comparison — Average Weekly Revenue per $1 Spend', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('images/phase5_roi_bar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

---## 9. Budget OptimizationUse `scipy.optimize.minimize` to find the allocation that maximizes predicted revenue under the same total budget.

In [ ]:
# Prediction function for a given budget allocation
def predict_revenue(allocation):
    """Given a 5-element budget allocation, predict weekly revenue."""
    features = {}
    for i, ch in enumerate(CHANNELS):
        p = CHANNEL_PARAMS[ch]
        spend_array = np.full(N_WEEKS, allocation[i])
        _, _, sat = apply_pipeline(spend_array, p['decay'], p['K'], p['S'])
        features[f'{ch}_saturated'] = sat.mean()

    # Use mean values for controls
    features['seasonality'] = df['seasonality'].mean()
    features['holiday_lift'] = df['holiday_lift'].mean()
    features['competitor_promo'] = df['competitor_promo'].mean()

    pred = model.params['const']
    for col in FEATURE_COLS:
        pred += model.params[col] * features[col]
    return pred

# Current allocation
current = np.array([spend_means[ch] for ch in CHANNELS])
total_budget = current.sum()
current_revenue = predict_revenue(current)

print(f"Current weekly budget: ${total_budget:,.0f}")
print(f"Current predicted revenue: ${current_revenue:,.0f}")

In [ ]:
# Optimize
def neg_revenue(alloc):
    return -predict_revenue(alloc)

bounds = [(current[i] * 0.3, current[i] * 3.0) for i in range(len(CHANNELS))]
constraint = {'type': 'eq', 'fun': lambda x: x.sum() - total_budget}

result = optimize.minimize(neg_revenue, current, method='SLSQP', bounds=bounds, constraints=constraint)
optimal = result.x
optimal_revenue = predict_revenue(optimal)

print("Optimization result:", "SUCCESS" if result.success else "FAILED")
print(f"\nOptimal predicted revenue: ${optimal_revenue:,.0f}")
print(f"Revenue lift: ${optimal_revenue - current_revenue:,.0f}/week = ${(optimal_revenue - current_revenue)*52:,.0f}/year")
print(f"Lift %: +{(optimal_revenue - current_revenue) / current_revenue * 100:.1f}%")

print(f"\n{'Channel':15s} {'Current':>10s} {'Optimal':>10s} {'Change':>10s}")
print("-" * 50)
for i, ch in enumerate(CHANNELS):
    label = CHANNEL_LABELS[f'{ch}_saturated']
    print(f"{label:15s} ${current[i]:>9,.0f} ${optimal[i]:>9,.0f} ${optimal[i]-current[i]:>+9,.0f}")

---## 10. Scenario ComparisonCompare Current vs. Optimal vs. a What-If scenario (cut TV by 50%, redistribute to digital).

In [ ]:
# What-If: Cut TV by 50%, redistribute proportionally to Search, Social, Display
tv_cut = current[0] * 0.50
whatif = current.copy()
whatif[0] -= tv_cut  # TV
digital_idx = [1, 2, 4]  # Search, Social, Display
for idx in digital_idx:
    whatif[idx] += tv_cut / len(digital_idx)
whatif_revenue = predict_revenue(whatif)

scenarios = {
    'Current': {'alloc': current, 'revenue': current_revenue},
    'Optimal': {'alloc': optimal, 'revenue': optimal_revenue},
    'What-If': {'alloc': whatif, 'revenue': whatif_revenue},
}

# Print comparison table
print(f"{'Channel':15s} {'Current':>10s} {'Optimal':>10s} {'What-If':>10s}")
print("-" * 50)
for i, ch in enumerate(CHANNELS):
    label = CHANNEL_LABELS[f'{ch}_saturated']
    print(f"{label:15s} ${current[i]:>9,.0f} ${optimal[i]:>9,.0f} ${whatif[i]:>9,.0f}")
print("-" * 50)
print(f"{'TOTAL':15s} ${current.sum():>9,.0f} ${optimal.sum():>9,.0f} ${whatif.sum():>9,.0f}")
print(f"{'Revenue':15s} ${current_revenue:>9,.0f} ${optimal_revenue:>9,.0f} ${whatif_revenue:>9,.0f}")
print(f"{'Annual Lift':15s} {'$0':>10s} ${(optimal_revenue-current_revenue)*52:>9,.0f} ${(whatif_revenue-current_revenue)*52:>9,.0f}")

In [ ]:
# Scenario comparison bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), gridspec_kw={'width_ratios': [3, 1.2]})
labels = [CHANNEL_LABELS[f'{ch}_saturated'] for ch in CHANNELS]
x = np.arange(len(CHANNELS))
w = 0.25

for i, (name, sc) in enumerate(scenarios.items()):
    bars = ax1.bar(x + i*w, sc['alloc']/1000, w, label=name)
    for bar, val in zip(bars, sc['alloc']):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'${val/1000:.0f}K', ha='center', va='bottom', fontsize=8)

ax1.set_xticks(x + w)
ax1.set_xticklabels(labels)
ax1.set_ylabel('Weekly Spend ($K)')
ax1.set_title('Budget Allocation by Scenario', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Revenue comparison
rev_vals = [sc['revenue'] for sc in scenarios.values()]
colors = ['#1f77b4', '#2ca02c', '#ff7f0e']
bars2 = ax2.bar(list(scenarios.keys()), rev_vals, color=colors, width=0.5)
for bar, val in zip(bars2, rev_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
             f'${val:,.0f}', ha='center', fontweight='bold', fontsize=10)
ax2.set_ylabel('Predicted Weekly Revenue ($)')
ax2.set_title('Revenue by Scenario', fontsize=14, fontweight='bold')
ax2.set_ylim(min(rev_vals)*0.95, max(rev_vals)*1.05)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('images/phase6_scenario_comparison_nb.png', dpi=150, bbox_inches='tight')
plt.show()

---## Summary of Key Findings| Metric | Value ||--------|-------|| Mean Weekly Revenue | $301,404 || Base (Organic) Share | 64% || Highest ROI Channel | Email (7.48x) || Lowest ROI Channel | TV (0.55x) || Optimal Revenue Lift | +$635K/year || Budget Change Required | $0 |### Recommendations1. **Cut TV** from $39K to $28K/week — deep in diminishing returns at 0.55x ROI2. **Scale Display** from $10K to $16K/week — 2.34x ROI with room to grow3. **Increase Search** to $24K/week — solid 1.15x ROI, can absorb more spend4. **Grow Email** from $3K to $5K/week — highest ROI at 7.48x, constrained by list size